# **[_Adding Metadata Columns During Ingestion_](url)**

We will explore how to add metadata columns during data ingestion.

In [0]:
%sql
SELECT current_catalog() as catalog, current_schema() as schema

In [0]:
%sql
USE CATALOG pysaprk_demo;

USE SCHEMA demo;

--  
SELECT current_catalog() as catalog, current_schema() as schema

### **_[Explore the Users Data](url)_**

/Volumes/pysaprk_demo/default/users_parque

In [0]:
spark.createDataFrame(dbutils.fs.ls('/Volumes/pysaprk_demo/default/users_parque')).show()

#### **_[Adding Metadata Columns to the Bronze Table During Ingestion](url)_**

When we ingesting data into the Bronze layer, you can apply transformations during ingestion and also retrieve metadata about the input files using _metadata column.

The _metadata column is a hidden column available for all supported file formats. To include it in the returned data, you must explicitly select it in the ready query that specifies the source.

#### **_[SQL Method](url)_**

In [0]:
%sql
SELECT 
  *,
  _metadata.file_name as source_file_name, -- Ingest data source file name
  _metadata.file_path as source_file_path, -- Ingest data source file path
  _metadata.file_modification_time as source_file_modification_time, -- Ingest data source file modification time
  _metadata.file_size as source_file_size, -- Ingest data source file size
  current_date() as current_date -- Ingest data source file current date
FROM read_files(
  '/Volumes/pysaprk_demo/default/users_parque',
  format => 'parquet'
)
limit 2
;

In [0]:
%sql
-- Drop the table if it already exists to start from scratch
DROP TABLE IF EXISTS tb_historical_users_bronze;

-- Create an empty table
CREATE TABLE IF NOT EXISTS tb_historical_users_bronze
SELECT 
  *,
  _metadata.file_name as source_file_name, -- Ingest data source file name
  _metadata.file_path as source_file_path, -- Ingest data source file path
  _metadata.file_modification_time as source_file_modification_time, -- Ingest data source file modification time
  _metadata.file_size as source_file_size, -- Ingest data source file size
  current_date() as current_date -- Ingest data source file current date
FROM read_files(
  '/Volumes/pysaprk_demo/default/users_parque',
  format => 'parquet'
);

-- View the final bronze table
SELECT * FROM tb_historical_users_bronze
limit 10;

-- Count the number of files ingested
SELECT source_file_name, count(*)
from tb_historical_users_bronze
group by source_file_name
;

### **_[Python Method](url)_**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# 1. Read the parquet file in cloud storage into a spark dataframe
df = (
  spark.read.parquet('/Volumes/pysaprk_demo/default/users_parque')
)

# 2. Add a matadata columns
df_with_metadata = (
  df.withColumn('source_file_name', col('_metadata.file_name'))
    .withColumn('source_file_path', col('_metadata.file_path'))
    .withColumn("source_file_size", col('_metadata.file_size'))
    .withColumn("ingestion_time", current_timestamp())
)

# 3. Write the dataframe to a delta table
(
  df_with_metadata.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(f'tb_historical_users_bronze_py')
) 


In [0]:
%sql
-- View the final bronze table
SELECT * FROM tb_historical_users_bronze_py
limit 2;

In [0]:
%sql
-- Count the number of files ingested
SELECT source_file_name, count(*)
from tb_historical_users_bronze_py
group by source_file_name
;